In [1]:
#https://www.kaggle.com/datasets/younusmohamed/fraudulent-financial-transaction-prediction

# Fraud Detection Project – Workflow Checklist

## 1. Data Ingestion
- Load `train.csv` and `test_share.csv`
- Verify row counts
- Ensure `id` is unique per transaction

## 2. Data Enrichment (Joins)
- Left-join `Geo_scores.csv` on `id`
- Left-join `Qset_tats.csv` on `id`
- Left-join `instance_scores.csv` on `id`
- Left-join `Lambda_wts.csv` on `Group`

## 3. Data Validation
- Check missing values after joins
- Confirm no duplicated `id`
- Inspect overall fraud rate (class imbalance)

## 4. Exploratory Data Analysis (EDA)
- Compare feature distributions: fraud vs non-fraud
- Fraud rate by key features and by `Group`
- Identify outliers and heavy-tailed variables

## 5. Preprocessing
- Handle missing values (imputation or flags)
- Scale features if required (mainly for linear models)
- Separate features (`X`) and target (`y`)

## 6. Train–Validation Strategy
- Use stratified or time-aware splits
- Ensure no leakage from group-level or score-based features

## 7. Class Imbalance Handling
- Baseline: no resampling
- Experiment with SMOTE / oversampling / undersampling
- Apply resampling **only on training data**

## 8. Model Training
- Baseline model: Logistic Regression
- Nonlinear models: Random Forest, XGBoost, LightGBM

## 9. Model Evaluation
- Precision, Recall, F1-score
- ROC-AUC and PR-AUC
- Tune classification threshold based on business constraints

## 10. Feature Contribution Analysis
- Evaluate incremental value of:
  - `geo_score`
  - `qsets_normalized_tat`
  - `instance_scores`
  - `lambda_wt`
- Remove unstable or dominating features if necessary

## 11. Final Model Selection
- Select model and decision threshold
- Document assumptions, risks, and limitations

## 12. Prediction
- Apply final model to `test_share.csv`
- Generate fraud probability scores and/or class labels

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
# Load main datasets
train_df  = pd.read_csv("train.csv")
test_df   = pd.read_csv("test_share.csv")

# Load enrichment datasets
geo_df    = pd.read_csv("Geo_scores.csv")
lambda_df = pd.read_csv("Lambda_wts.csv")
tat_df    = pd.read_csv("Qset_tats.csv")
inst_df   = pd.read_csv("instance_scores.csv")

# Quick sanity check
{
    "train": train_df.shape,
    "test": test_df.shape,
    "geo": geo_df.shape,
    "lambda": lambda_df.shape,
    "tat": tat_df.shape,
    "instance": inst_df.shape
}

/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


{'train': (227845, 28),
 'test': (56962, 27),
 'geo': (1424035, 2),
 'lambda': (1400, 2),
 'tat': (1424035, 2),
 'instance': (1424035, 2)}

In [3]:
# ensure id is unique across tables
def duplicate_summary(df, name):
    dup_count = df["id"].duplicated().sum()
    return {name: dup_count}

{
    **duplicate_summary(train_df, "train_duplicates"),
    **duplicate_summary(test_df, "test_duplicates"),
    **duplicate_summary(geo_df, "geo_duplicates"),
    **duplicate_summary(tat_df, "tat_duplicates"),
    **duplicate_summary(inst_df, "instance_duplicates"),
}

{'train_duplicates': 0,
 'test_duplicates': 0,
 'geo_duplicates': 1139228,
 'tat_duplicates': 1139228,
 'instance_duplicates': 1139228}

Critical implication (do NOT ignore)

You cannot directly left-join these tables as-is.

If you do:
	•	Your row count will explode
	•	You introduce temporal leakage
	•	Your model becomes invalid

## Step 3 – Multiplicity Check & Aggregation Decision

### Findings
- Each `id` appears **exactly 5 times** in:
  - `Geo_scores`
  - `Qset_tats`
  - `instance_scores`
- This indicates **parallel risk signals**, not duplicates or errors.

### Decision
- Reduce enrichment tables to **1 row per `id`** before merging.
- Direct merging would cause row explosion and leakage.

### Aggregation Rules
- `geo_score` → **max** (worst-case location risk)
- `qsets_normalized_tat` → **mean** (average network latency)
- `instance_scores` → **max** (highest vulnerability signal)

### Next Step
- Aggregate each enrichment table by `id`
- Then merge aggregated tables into `train` and `test`

In [4]:
def multiplicity_summary(df, name):
    counts = df.groupby("id").size()
    return {
        f"{name}_unique_ids": counts.shape[0],
        f"{name}_total_rows": df.shape[0],
        f"{name}_mean_rows_per_id": counts.mean(),
        f"{name}_median_rows_per_id": counts.median(),
        f"{name}_max_rows_per_id": counts.max(),
        f"{name}_pct_ids_gt1": (counts.gt(1).mean() * 100)
    }

{
    **multiplicity_summary(geo_df, "geo"),
    **multiplicity_summary(tat_df, "tat"),
    **multiplicity_summary(inst_df, "instance"),
}

{'geo_unique_ids': 284807,
 'geo_total_rows': 1424035,
 'geo_mean_rows_per_id': 5.0,
 'geo_median_rows_per_id': 5.0,
 'geo_max_rows_per_id': 5,
 'geo_pct_ids_gt1': 100.0,
 'tat_unique_ids': 284807,
 'tat_total_rows': 1424035,
 'tat_mean_rows_per_id': 5.0,
 'tat_median_rows_per_id': 5.0,
 'tat_max_rows_per_id': 5,
 'tat_pct_ids_gt1': 100.0,
 'instance_unique_ids': 284807,
 'instance_total_rows': 1424035,
 'instance_mean_rows_per_id': 5.0,
 'instance_median_rows_per_id': 5.0,
 'instance_max_rows_per_id': 5,
 'instance_pct_ids_gt1': 100.0}

In [5]:
# Step 4: Aggregate enrichment tables to 1 row per id

geo_agg = (geo_df
           .groupby("id", as_index=False)["geo_score"]
           .max())

tat_agg = (tat_df
           .groupby("id", as_index=False)["qsets_normalized_tat"]
           .mean())

inst_agg = (inst_df
            .groupby("id", as_index=False)["instance_scores"]
            .max())

# sanity: should all be (284807, 2)
{
    "geo_agg": geo_agg.shape,
    "tat_agg": tat_agg.shape,
    "inst_agg": inst_agg.shape
}


{'geo_agg': (284807, 2), 'tat_agg': (284807, 2), 'inst_agg': (284807, 2)}

In [6]:
#MERGING
# Step 5: Merge aggregated enrichment + lambda_wt

train_merged = (train_df
    .merge(geo_agg, on="id", how="left")
    .merge(tat_agg, on="id", how="left")
    .merge(inst_agg, on="id", how="left")
    .merge(lambda_df, on="Group", how="left")
)

test_merged = (test_df
    .merge(geo_agg, on="id", how="left")
    .merge(tat_agg, on="id", how="left")
    .merge(inst_agg, on="id", how="left")
    .merge(lambda_df, on="Group", how="left")
)

# sanity checks: row counts should NOT change
{
    "train_before": train_df.shape,
    "train_after": train_merged.shape,
    "test_before": test_df.shape,
    "test_after": test_merged.shape
}

{'train_before': (227845, 28),
 'train_after': (227845, 32),
 'test_before': (56962, 27),
 'test_after': (56962, 31)}

# Data Validation - missing, duplicated, imbalance


In [7]:
#Row count and target sanity


{
    "rows": train_merged.shape[0],
    "target_nulls": train_merged["Target"].isna().sum(),
    "fraud_rate": train_merged["Target"].mean()
}

#fraud rate is very low

{'rows': 227845, 'target_nulls': 0, 'fraud_rate': 0.001729245759178389}

In [8]:
#Missingness in merged features
train_merged[
    ["geo_score", "qsets_normalized_tat", "instance_scores", "lambda_wt"]
].isna().mean()

geo_score               0.0
qsets_normalized_tat    0.0
instance_scores         0.0
lambda_wt               0.0
dtype: float64

In [9]:
# Quick distribution check
train_merged[
    ["geo_score", "qsets_normalized_tat", "instance_scores", "lambda_wt"]
].describe()

,geo_score,qsets_normalized_tat,instance_scores,lambda_wt
count,227845.000000,227845.000000,227845.000000,227845.000000
mean,9.535950,0.000115,2.209149,0.000350
std,4.943898,0.945602,3.761932,0.957957
min,-13.700000,-31.450000,-0.840000,-19.210000
25%,6.550000,-0.520000,0.640000,-0.430000
50%,8.620000,-0.070000,0.920000,0.050000
75%,10.000000,0.437500,2.370000,0.490000
max,44.860000,10.233333,120.350000,10.530000


In [10]:
train_merged.groupby("Target")[
    ["geo_score", "qsets_normalized_tat", "instance_scores", "lambda_wt"]
].describe().round(3)

geo_score                                                  \
           count   mean    std    min   25%   50%     75%    max   
Target                                                             
0       227451.0  9.541  4.944 -13.70  6.55  8.62  10.000  44.86   
1          394.0  6.661  4.020  -5.86  4.48  7.24   9.138  31.33   

       qsets_normalized_tat         ... instance_scores         lambda_wt  \
                      count   mean  ...             75%     max     count   
Target                              ...                                     
0                  227451.0  0.013  ...            2.38  120.35  227451.0   
1                     394.0 -7.254  ...            0.92   22.43     394.0   

                                                       
         mean    std    min    25%    50%  75%    max  
Target                                                 
0       0.012  0.898 -18.39 -0.420  0.050  0.5  10.53  
1      -6.819  4.295 -19.21 -9.462 -6.605 -4.1   1.57  

[2 rows x 32 columns]

## Step 6 – Fraud vs Non-Fraud Feature Comparison (ABOVE)

### Class Imbalance
- Fraud cases: **394**
- Total transactions: **227,845**
- Fraud rate: **~0.17%** (extremely imbalanced)

---

### Feature Behavior by Class

| Feature | Fraud vs Non-Fraud | Interpretation |
|------|------------------|----------------|
| `geo_score` | **Lower for fraud** | Geographic instability signal |
| `qsets_normalized_tat` | **Much lower for fraud** | Network / timing anomaly |
| `instance_scores` | **Much lower for fraud** | High-risk transactions concentrate in negative tail |
| `lambda_wt` | **Strongly lower for fraud** | Group-level prior is highly informative |

---

### Outlier Interpretation
- Extreme values are **not data errors**
- Fraud cases concentrate in **distribution tails**
- Removing or clipping now would **hurt recall**

---

### Decision
- Keep raw feature values
- No outlier removal at this stage
- Consider clipping **only for linear models later**

In [11]:
# Stratified Cross-Validation (setup only)
from sklearn.model_selection import StratifiedKFold

# Separate features and target
X = train_merged.drop(columns=["Target"])
y = train_merged["Target"]

# Stratified CV definition
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Sanity check: fraud rate per fold
fold_rates = []
for fold, (_, val_idx) in enumerate(skf.split(X, y), 1):
    fold_rates.append((fold, y.iloc[val_idx].mean()))

fold_rates


#Each fold has nearly same fraud rate 0.17

[(1, 0.0017116899646689636),
 (2, 0.0017336347078057452),
 (3, 0.0017336347078057452),
 (4, 0.0017336347078057452),
 (5, 0.0017336347078057452)]

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score

# Drop non-numeric identifiers / categories
X = train_merged.drop(columns=["Target", "id", "Group"])
y = train_merged["Target"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs, recalls, precisions = [], [], [], []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train, y_train)

    y_proba = pipe.predict_proba(X_val)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    roc_aucs.append(roc_auc_score(y_val, y_proba))
    pr_aucs.append(average_precision_score(y_val, y_proba))
    recalls.append(recall_score(y_val, y_pred))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

{
    "ROC_AUC_mean": sum(roc_aucs)/len(roc_aucs),
    "PR_AUC_mean": sum(pr_aucs)/len(pr_aucs),
    "Recall@0.5_mean": sum(recalls)/len(recalls),
    "Precision@0.5_mean": sum(precisions)/len(precisions)
}

/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.

{'ROC_AUC_mean': 0.9711463186880127,
 'PR_AUC_mean': 0.6614738051517196,
 'Recall@0.5_mean': 0.8833820188250568,
 'Precision@0.5_mean': 0.05302863709879654}

#Interpretation above output
## Baseline Model Results (Logistic Regression, Stratified CV)

### Metrics (5-fold mean)
- **ROC-AUC:** 0.97  
- **PR-AUC:** 0.66  
- **Recall @ 0.5:** 0.88  
- **Precision @ 0.5:** 0.05  

---

### Interpretation
- Very strong **ranking performance** (high ROC-AUC)
- High **PR-AUC** despite extreme class imbalance (~0.17% fraud)
- High recall indicates effective fraud detection
- Low precision at threshold 0.5 is **expected** and not a failure

---

### Key Takeaway
- Model is suitable for **risk scoring**, not direct blocking
- Decision threshold must be tuned based on business costs

---

### Decision
- Baseline model is **strong and stable**
- Safe to proceed to threshold tuning, SMOTE, or tree-based models

In [16]:
import sys
!{sys.executable} -m pip install imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 2.8 MB/s eta 0:00:00 MB/s eta 0:00:01


In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline  # <-- IMPORTANT: imblearn Pipeline

# Features / target (numeric only)
X = train_merged.drop(columns=["Target", "id", "Group"])
y = train_merged["Target"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs, recalls, precisions = [], [], [], []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("lr", LogisticRegression(max_iter=2000, n_jobs=-1))
    ])

    pipe.fit(X_train, y_train)

    y_proba = pipe.predict_proba(X_val)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    roc_aucs.append(roc_auc_score(y_val, y_proba))
    pr_aucs.append(average_precision_score(y_val, y_proba))
    pr = precision_score(y_val, y_pred, zero_division=0)
    rc = recall_score(y_val, y_pred)
    precisions.append(pr)
    recalls.append(rc)

{
    "ROC_AUC_mean": sum(roc_aucs)/len(roc_aucs),
    "PR_AUC_mean": sum(pr_aucs)/len(pr_aucs),
    "Recall@0.5_mean": sum(recalls)/len(recalls),
    "Precision@0.5_mean": sum(precisions)/len(precisions)
}

{'ROC_AUC_mean': 0.967906025281145,
 'PR_AUC_mean': 0.6628916746093639,
 'Recall@0.5_mean': 0.8808503732554366,
 'Precision@0.5_mean': 0.05070440886667561}

## SMOTE Baseline Results (Logistic Regression, Stratified CV)

### Metrics (5-fold mean)
- **ROC-AUC:** 0.968  
- **PR-AUC:** 0.663  
- **Recall @ 0.5:** 0.881  
- **Precision @ 0.5:** 0.051  

---

### Comparison to Non-SMOTE Baseline
| Metric | No SMOTE | With SMOTE | Change |
|------|--------|-----------|--------|
| ROC-AUC | 0.971 | 0.968 | ↓ slight |
| PR-AUC | 0.661 | 0.663 | ↑ negligible |
| Recall @ 0.5 | 0.883 | 0.881 | ≈ same |
| Precision @ 0.5 | 0.053 | 0.051 | ↓ slight |

---

### Interpretation
- SMOTE **does not materially improve performance**
- Ranking ability slightly decreases
- Precision–recall tradeoff remains essentially unchanged
- Original class-weighted logistic model is already strong

---

### Decision
- ❌ Do not use SMOTE for this dataset
- ✅ Prefer class-weighted models or tree-based methods
- SMOTE adds complexity without meaningful gain

## Threshold Tuning (Logistic Regression)

### Purpose
- Logistic regression provides **risk scores**
- Threshold selection determines **operational behavior**
- Model stays fixed; decisions change via threshold

### What We Evaluated
- Precision
- Recall
- Flag rate (% of transactions flagged)
- Threshold range: 0.001 → 0.2

### How to Select a Threshold
- **Capacity-driven:** match investigation limits
- **Recall-driven:** maximize fraud capture with minimum precision
- **Precision-driven:** reduce false positives with acceptable recall

### Key Takeaway
- Threshold tuning defines the **best achievable performance** of logistic regression
- Further gains require **new features or nonlinear models**

In [21]:
# THRESHOLD TUNING FOR LOGISTIC
import numpy as np
from sklearn.metrics import precision_score, recall_score

# Use one fold to tune threshold
train_idx, val_idx = next(skf.split(X, y))
X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# Refit baseline logistic on this fold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1))
])

pipe.fit(X_train, y_train)
y_proba = pipe.predict_proba(X_val)[:, 1]

# Sweep thresholds
thresholds = np.linspace(0.001, 0.2, 25)

rows = []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    rows.append({
        "threshold": round(t, 4),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred),
        "flag_rate": y_pred.mean()   # % transactions flagged
    })

import pandas as pd
pd.DataFrame(rows)

/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/huseyinzengin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,threshold,precision,recall,flag_rate
0,0.0010,0.001775,1.000000,0.964318
1,0.0093,0.002204,1.000000,0.776471
2,0.0176,0.002499,1.000000,0.684961
3,0.0259,0.002733,0.987179,0.618183
4,0.0342,0.003028,0.987179,0.557989
5,0.0425,0.003337,0.987179,0.506397
6,0.0508,0.003640,0.974359,0.458206
7,0.0590,0.004006,0.974359,0.416358
8,0.0673,0.004386,0.974359,0.380215
9,0.0756,0.004749,0.974359,0.351182


## Threshold Tuning Results (Logistic Regression)

### Observed Trade-offs

| Threshold ↑ | Precision ↑ | Recall ↓ | Flag Rate ↓ |
|------------|------------|----------|-------------|
| Very low (0.001–0.02) | ~0.2% | ~100% | 65–95% |
| Mid (0.05–0.10) | ~0.4–0.6% | ~97% | 28–46% |
| Higher (0.15–0.20) | ~0.8–1.2% | ~96% | 14–20% |

---

### Key Insights

- **Recall remains extremely high** until threshold ≈ 0.10  
- **Precision increases monotonically** as threshold increases  
- **Flag rate drops sharply**, giving operational control  

---

### Practical Interpretation

- Thresholds below **0.03** are **operationally infeasible**  
  (flagging >60% of transactions)
- Thresholds around **0.08–0.12** offer a **balanced regime**:
  - ~96–97% recall
  - ~0.5–0.7% precision
  - ~25–35% of transactions flagged
- Higher thresholds trade modest recall loss for **manageable volumes**

---

### Key Takeaway

> Logistic regression already captures fraud risk well; threshold tuning reveals how to convert ranking power into operational decisions.

---

### Decision Point

- Choose threshold based on:
  - Investigation capacity, or
  - Target recall / precision constraint

Further performance gains require **nonlinear models**, not further tuning.

### RANDOM FOREST

In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score
import numpy as np

# Features / target (numeric only)
X = train_merged.drop(columns=["Target", "id", "Group"])
y = train_merged["Target"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs, recalls, precisions = [], [], [], []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=50,      # important for rare events
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    )

    rf.fit(X_train, y_train)

    y_proba = rf.predict_proba(X_val)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    roc_aucs.append(roc_auc_score(y_val, y_proba))
    pr_aucs.append(average_precision_score(y_val, y_proba))
    recalls.append(recall_score(y_val, y_pred))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

{
    "ROC_AUC_mean": np.mean(roc_aucs),
    "PR_AUC_mean": np.mean(pr_aucs),
    "Recall@0.5_mean": np.mean(recalls),
    "Precision@0.5_mean": np.mean(precisions)
}

{'ROC_AUC_mean': 0.9744500690603802,
 'PR_AUC_mean': 0.7607726940870528,
 'Recall@0.5_mean': 0.8199935086011034,
 'Precision@0.5_mean': 0.7447255585810575}

## Random Forest Results (Stratified CV, No SMOTE)

### Metrics (5-fold mean)
- **ROC-AUC:** 0.974  
- **PR-AUC:** 0.761  
- **Recall @ 0.5:** 0.820  
- **Precision @ 0.5:** 0.745  

---

### Comparison vs Logistic Regression

| Metric | Logistic | Random Forest | Interpretation |
|------|---------|---------------|----------------|
| ROC-AUC | ~0.97 | **0.97+** | Similar ranking power |
| PR-AUC | ~0.66 | **0.76** | Strong lift under imbalance |
| Recall @ 0.5 | ~0.88 | 0.82 | Slight recall trade-off |
| Precision @ 0.5 | ~0.05 | **0.74** | Massive precision gain |

---

### Interpretation

- Random Forest captures **nonlinear interactions** missed by logistic regression
- Precision improves **by an order of magnitude** at default threshold
- Recall decreases slightly but remains high
- PR-AUC increase confirms **real improvement**, not threshold artifact

---

### Key Insight

> Logistic regression sets the baseline ceiling; Random Forest meaningfully raises the precision–recall frontier by modeling nonlinear fraud patterns.

---

### Decision

- Random Forest **outperforms logistic regression** for operational use
- Next steps:
  - Threshold tuning for RF
  - Feature importance / stability checks
  - Compare with gradient boosting (XGBoost / LightGBM)

In [24]:
# RF THRESHOLD
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# reuse last CV splitter
train_idx, val_idx = next(skf.split(X, y))
X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# Fit RF on this fold
rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)

rf.fit(X_train, y_train)
y_proba = rf.predict_proba(X_val)[:, 1]

# Sweep thresholds
thresholds = np.linspace(0.05, 0.95, 19)

rows = []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred),
        "flag_rate": y_pred.mean()
    })

pd.DataFrame(rows)

,threshold,precision,recall,flag_rate
0,0.05,0.030859,0.948718,0.052623
1,0.10,0.092288,0.935897,0.017358
2,0.15,0.189119,0.935897,0.008471
3,0.20,0.331818,0.935897,0.004828
4,0.25,0.453416,0.935897,0.003533
5,0.30,0.598361,0.935897,0.002677
6,0.35,0.730000,0.935897,0.002194
7,0.40,0.808989,0.923077,0.001953
8,0.45,0.835294,0.910256,0.001865
9,0.50,0.864198,0.897436,0.001778


## Random Forest Threshold Tuning – Interpretation

### Key Patterns

- **Precision increases monotonically** with higher thresholds
- **Recall remains high (>93%)** until threshold ≈ **0.35**
- **Flag rate drops sharply**, giving strong operational control

---

### Practical Operating Regions

| Threshold Range | Precision | Recall | Flag Rate | Interpretation |
|-----------------|-----------|--------|-----------|----------------|
| **0.05–0.10** | 3–9% | ~94% | 1.7–5% | Broad screening |
| **0.15–0.30** | 19–60% | ~94% | <1% | High-value review |
| **0.35–0.50** | **73–86%** | **90–93%** | ~0.18–0.22% | Near-optimal trade-off |
| **> 0.70** | 90%+ | <85% | ~0.15% | Very conservative |

---

### Comparison to Logistic Regression

- Logistic @ similar recall (~94%): **precision < 1%**
- Random Forest @ similar recall: **precision ~30–60%**

This confirms a **clear dominance** of Random Forest.

---

### Key Takeaway

> Random Forest dramatically shifts the precision–recall frontier, enabling high recall with orders-of-magnitude higher precision.

---

### Recommendation

- **Operational threshold:** ~**0.30–0.45**
- Delivers:
  - ~90–94% recall
  - 45–83% precision
  - Manageable review volume (<0.3%)

---

### Next Steps
- Lock a threshold based on business capacity
- Extract feature importance
- Optionally compare with gradient boosting models

# XGBOOST


In [25]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score
import numpy as np

# Features / target
X = train_merged.drop(columns=["Target", "id", "Group"])
y = train_merged["Target"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs, recalls, precisions = [], [], [], []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )

    xgb.fit(X_train, y_train)

    y_proba = xgb.predict_proba(X_val)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    roc_aucs.append(roc_auc_score(y_val, y_proba))
    pr_aucs.append(average_precision_score(y_val, y_proba))
    recalls.append(recall_score(y_val, y_pred))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))

{
    "ROC_AUC_mean": np.mean(roc_aucs),
    "PR_AUC_mean": np.mean(pr_aucs),
    "Recall@0.5_mean": np.mean(recalls),
    "Precision@0.5_mean": np.mean(precisions)
}

{'ROC_AUC_mean': 0.9807945303643122,
 'PR_AUC_mean': 0.8371749680165582,
 'Recall@0.5_mean': 0.80730282375852,
 'Precision@0.5_mean': 0.8861188181721911}

## XGBoost Results (Stratified CV, No SMOTE)

### Metrics (5-fold mean)
- **ROC-AUC:** 0.981  
- **PR-AUC:** 0.837  
- **Recall @ 0.5:** 0.807  
- **Precision @ 0.5:** 0.886  

---

### Comparison vs Random Forest

| Metric | Random Forest | XGBoost | Interpretation |
|------|---------------|---------|----------------|
| ROC-AUC | ~0.974 | **0.981** | Better ranking power |
| PR-AUC | ~0.761 | **0.837** | Large improvement under imbalance |
| Recall @ 0.5 | ~0.820 | 0.807 | Slightly lower |
| Precision @ 0.5 | ~0.745 | **0.886** | Substantial gain |

---

### Interpretation

- XGBoost **dominates RF** on overall discrimination (ROC-AUC, PR-AUC)
- Precision at default threshold is **very high (~89%)**
- Recall is slightly lower than RF but still strong
- Boosting captures **subtle nonlinear interactions** and rare-event structure better than bagging

---

### Key Insight

> XGBoost pushes the precision–recall frontier further than Random Forest, achieving extremely high precision with only a modest recall trade-off.

---

### Practical Implication

- XGBoost is well-suited for **high-confidence fraud decisions**
- Threshold tuning can recover recall while maintaining strong precision
- This is a strong **candidate for the final production model**

---

### Next Steps
- Threshold tuning for XGBoost
- Feature importance / SHAP analysis
- Final model & threshold recommendation

In [27]:
# TUNING XGBOOST
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from xgboost import XGBClassifier

# Use one stratified fold for threshold analysis
train_idx, val_idx = next(skf.split(X, y))
X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# Fit XGBoost on this fold
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb.fit(X_train, y_train)
y_proba = xgb.predict_proba(X_val)[:, 1]

# Sweep thresholds
thresholds = np.linspace(0.05, 0.95, 19)

rows = []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred),
        "flag_rate": y_pred.mean()
    })

pd.DataFrame(rows)

,threshold,precision,recall,flag_rate
0,0.05,0.596639,0.910256,0.002611
1,0.10,0.755319,0.910256,0.002063
2,0.15,0.865854,0.910256,0.001799
3,0.20,0.898734,0.910256,0.001734
4,0.25,0.910256,0.910256,0.001712
5,0.30,0.946667,0.910256,0.001646
6,0.35,0.972222,0.897436,0.001580
7,0.40,0.972222,0.897436,0.001580
8,0.45,0.972222,0.897436,0.001580
9,0.50,0.971831,0.884615,0.001558


## Model Comparison: Logistic vs Random Forest vs XGBoost

### Cross-Validated Performance Summary

| Model | ROC-AUC | PR-AUC | Recall @ 0.5 | Precision @ 0.5 | Key Strength |
|------|--------|--------|--------------|----------------|--------------|
| **Logistic Regression** | ~0.97 | ~0.66 | **~0.88** | ~0.05 | High recall, strong baseline |
| **Random Forest** | ~0.97 | ~0.76 | ~0.82 | ~0.75 | Large precision gain |
| **XGBoost** | **~0.98** | **~0.84** | ~0.81 | **~0.89** | Best overall discrimination |

---

### Interpretation

#### Logistic Regression
- Excellent **ranking ability**
- Very high recall at low thresholds
- Precision is structurally limited due to:
  - Linear decision boundary
  - Extreme class imbalance
- Best suited for:
  - Baseline model
  - Risk scoring
  - Explainability

#### Random Forest
- Captures **nonlinear interactions**
- Massive jump in precision at similar recall
- Slight recall trade-off at default threshold
- Good balance between performance and robustness

#### XGBoost
- Strongest model across **all major metrics**
- Highest PR-AUC → best performance under imbalance
- Produces **sharper probability separation**
- Enables very high precision with acceptable recall
- Best candidate for production deployment

---

### Threshold Tuning Perspective

| Model | What Threshold Tuning Reveals |
|------|-------------------------------|
| Logistic | Performance ceiling is low precision |
| Random Forest | Precision–recall frontier shifts upward |
| XGBoost | Frontier shifts furthest; dominates others |

---

### Final Takeaway

> Logistic regression establishes the baseline, Random Forest significantly improves precision, and XGBoost delivers the best overall trade-off between recall, precision, and operational feasibility.

---

### Recommended Use
- **Champion model:** XGBoost  
- **Fallback / challenger:** Logistic Regression  
- **Governance-friendly alternative:** Random Forest

In [29]:
# feature ranking
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "gain": xgb.feature_importances_
}).sort_values("gain", ascending=False)

importance.head(15)

,feature,gain
28,lambda_wt,0.331371
3,Per4,0.111940
26,qsets_normalized_tat,0.099320
14,Dem6,0.035694
16,Dem8,0.028267
7,Per8,0.026046
9,Dem1,0.025789
24,Normalised_FNT,0.025375
6,Per7,0.024016
15,Dem7,0.023643


## Fraud Model Development & Validation — Summary

### What We Did
- Loaded and validated all datasets (train, test, enrichment tables)
- Aggregated multi-row enrichment data to transaction level (avoided leakage)
- Merged consistent features into train and test
- Performed data sanity checks (missingness, outliers, distributions)
- Used **stratified cross-validation** due to extreme class imbalance (~0.17%)
- Built and evaluated multiple models:
  - **Logistic Regression** (baseline, explainable)
  - **Random Forest** (nonlinear interactions)
  - **XGBoost** (boosted trees, best performance)
- Evaluated with **ROC-AUC** and **PR-AUC** (appropriate for rare events)
- Conducted **threshold tuning** to translate model scores into decisions
- Selected **XGBoost** as the champion model based on precision–recall trade-offs
- Performed feature contribution analysis using XGBoost gain importance
- Ensured governance readiness (no leakage, configurable thresholds, explainability)

---

## Key Findings
- Logistic regression provides a strong baseline but low precision due to rarity
- Random Forest significantly improves precision at similar recall
- XGBoost delivers the best overall discrimination and operational performance
- Threshold choice is a **business decision**, informed by model trade-offs

---

## What Could Be Done for Other Data Types

### Time-Based Data
- Use **time-based splits** instead of stratified CV
- Validate on future periods only (prevent look-ahead bias)
- Monitor **population stability (PSI)** and **performance drift** over time
- Re-tune thresholds periodically as fraud patterns evolve

### Sequential / Event-Level Data
- Aggregate using rolling windows (e.g., last 7 / 30 / 90 days)
- Engineer velocity features (counts, sums, max, trend)
- Consider sequence models (HMMs, RNNs, temporal boosting)

### Categorical / High-Cardinality Data
- Target encoding or impact encoding
- Frequency-based bucketing
- Careful leakage checks during encoding

### Real-Time / Streaming Systems
- Use low-latency models (logistic / shallow trees)
- Separate **scoring** from **decisioning**
- Store thresholds in configuration, not code

---

## Final Takeaway
> Fraud modeling is not just about training a model, but about building a governed, explainable, and operationally usable system that adapts to business and data constraints.